# Analysis Summary (No Plots)

This notebook provides paper-ready aggregate statistics from `outputs/aggregate_df_all.pkl`.

It focuses on:

1. Public/private divergence rates.
2. Cosine self-consistency summaries.
3. NLI entailment/neutral/contradiction summaries.

All sections are table/text based (no figures).

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


# Resolve project root whether notebook is run from repo root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / "outputs").exists() and (ROOT.parent / "outputs").exists():
    ROOT = ROOT.parent

AGGREGATE_PKL = ROOT / "outputs" / "aggregate_df_all.pkl"
if not AGGREGATE_PKL.is_file():
    raise FileNotFoundError(f"Missing aggregate pickle: {AGGREGATE_PKL}")

with AGGREGATE_PKL.open("rb") as f:
    aggregate_df = pickle.load(f)


def _is_missing(x) -> bool:
    if x is None:
        return True
    try:
        return bool(pd.isna(x))
    except Exception:
        return False


def incentive_bucket(direction, inc_type) -> str:
    """Collapse incentive settings into paper-facing groups."""
    if _is_missing(direction) and _is_missing(inc_type):
        return "baseline"

    d = str(direction).strip().lower() if not _is_missing(direction) else ""
    if d == "negative":
        return "persona_reinforcing"
    if d == "positive":
        return "alignment_inducing"
    return f"other::{direction}|{inc_type}"


BUCKET_ORDER = ["persona_reinforcing", "baseline", "alignment_inducing"]
BUCKET_LABEL = {
    "persona_reinforcing": "Persona-Reinforcing",
    "baseline": "Baseline",
    "alignment_inducing": "Alignment-Inducing",
}

aggregate_df = aggregate_df.copy()
aggregate_df["incentive_bucket"] = [
    incentive_bucket(d, t)
    for d, t in zip(aggregate_df["incentive_direction"], aggregate_df["incentive_type"])
]

print(f"Loaded {len(aggregate_df)} experiment rows from: {AGGREGATE_PKL}")
print("Incentive buckets:", aggregate_df["incentive_bucket"].value_counts(dropna=False).to_dict())

Loaded 150 experiment rows from: /Users/ourmangg/Documents/Personal_Project/LLMAgora/outputs/aggregate_df_all.pkl
Incentive buckets: {'persona_reinforcing': 60, 'alignment_inducing': 60, 'baseline': 30}


In [2]:
display(Markdown("## 1) Public/Private divergence summary (agent-specific)"))

DECISION_COL = "decision-self-consistency-all-repeats"


def build_divergence_trials(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        payload = row.get(DECISION_COL)
        if not isinstance(payload, dict):
            continue

        for rep in payload.get("repeats") or []:
            if not isinstance(rep, dict):
                continue
            rep_no = rep.get("repeat_number")
            for agent in ("alpha", "beta"):
                block = rep.get(agent) or {}
                pub = (block.get("public") or {}).get("decisions") or []
                prv = (block.get("private") or {}).get("decisions") or []
                turns = (block.get("public") or {}).get("turns") or list(range(1, min(len(pub), len(prv)) + 1))

                for ti, (a, b) in enumerate(zip(pub, prv)):
                    if a is None or b is None:
                        continue
                    turn = turns[ti] if ti < len(turns) else (ti + 1)
                    rows.append(
                        {
                            "model": str(row.get("model")),
                            "scenario_id": str(row.get("scenario_id")),
                            "incentive_bucket": row.get("incentive_bucket"),
                            "incentive_direction": row.get("incentive_direction"),
                            "incentive_type": row.get("incentive_type"),
                            "agent": agent,
                            "repeat_number": rep_no,
                            "debate_turn": int(turn),
                            "divergent": int(int(a) != int(b)),
                        }
                    )

    return pd.DataFrame(rows)


div_trials = build_divergence_trials(aggregate_df)
if div_trials.empty:
    raise ValueError("No divergence trials found from decision-self-consistency-all-repeats.")

# Agent-specific pooled summary.
div_overall = (
    div_trials.groupby(["agent", "incentive_bucket"], as_index=False)
    .agg(
        divergence_rate=("divergent", "mean"),
        divergence_se=("divergent", "sem"),
        divergent_trials=("divergent", "sum"),
        total_trials=("divergent", "size"),
        n_models=("model", "nunique"),
        n_scenarios=("scenario_id", "nunique"),
    )
)
div_overall["bucket_label"] = div_overall["incentive_bucket"].map(BUCKET_LABEL).fillna(div_overall["incentive_bucket"])
div_overall["divergence_rate_pct"] = 100.0 * div_overall["divergence_rate"]
div_overall["divergence_se_pct"] = 100.0 * div_overall["divergence_se"]

# Baseline deltas computed per agent.
base_map = (
    div_overall[div_overall["incentive_bucket"] == "baseline"]
    .set_index("agent")["divergence_rate_pct"]
    .to_dict()
)
div_overall["delta_vs_baseline_pct_points"] = [
    r.divergence_rate_pct - base_map.get(r.agent, np.nan)
    for r in div_overall.itertuples(index=False)
]

div_overall = div_overall.sort_values(["agent", "incentive_bucket"], kind="stable").reset_index(drop=True)

display(Markdown("### 1A. Agent-specific pooled divergence by incentive bucket"))
display(
    div_overall[
        [
            "agent",
            "bucket_label",
            "divergence_rate_pct",
            "divergence_se_pct",
            "delta_vs_baseline_pct_points",
            "divergent_trials",
            "total_trials",
            "n_models",
            "n_scenarios",
        ]
    ].round(3)
)

# Agent-specific model-balanced summary.
div_model_balanced = (
    div_trials.groupby(["agent", "model", "incentive_bucket"], as_index=False)
    .agg(model_divergence_rate=("divergent", "mean"))
    .groupby(["agent", "incentive_bucket"], as_index=False)
    .agg(
        model_balanced_divergence_rate=("model_divergence_rate", "mean"),
        model_balanced_divergence_se=("model_divergence_rate", "sem"),
        model_divergence_rate_std=("model_divergence_rate", "std"),
        n_models=("model_divergence_rate", "size"),
    )
)
div_model_balanced["model_balanced_divergence_rate_pct"] = 100.0 * div_model_balanced["model_balanced_divergence_rate"]
div_model_balanced["model_balanced_divergence_se_pct"] = 100.0 * div_model_balanced["model_balanced_divergence_se"]

display(Markdown("### 1B. Agent-specific model-balanced divergence by incentive bucket"))
display(
    div_model_balanced[[
        "agent",
        "incentive_bucket",
        "model_balanced_divergence_rate_pct",
        "model_balanced_divergence_se_pct",
        "model_divergence_rate_std",
        "n_models",
    ]].round(3)
)

display(Markdown("### 1C. P Divergence by model and incentive bucket (% divergent), split by agent"))
div_model_pivot = (
    div_trials.groupby(["model", "agent", "incentive_bucket"], as_index=False)
    .agg(divergence_rate=("divergent", "mean"))
)
div_model_pivot["divergence_rate_pct"] = 100.0 * div_model_pivot["divergence_rate"]
display(
    div_model_pivot.pivot(index=["model", "agent"], columns="incentive_bucket", values="divergence_rate_pct").round(3)
)

display(Markdown("### 1D. Divergence by scenario and incentive bucket (% divergent), split by agent"))
div_scenario_pivot = (
    div_trials.groupby(["scenario_id", "agent", "incentive_bucket"], as_index=False)
    .agg(divergence_rate=("divergent", "mean"))
)
div_scenario_pivot["divergence_rate_pct"] = 100.0 * div_scenario_pivot["divergence_rate"]
display(
    div_scenario_pivot.pivot(index=["scenario_id", "agent"], columns="incentive_bucket", values="divergence_rate_pct").round(3)
)

print(f"Divergence trial count used: {len(div_trials):,}")

## 1) Public/Private divergence summary (agent-specific)

### 1A. Agent-specific pooled divergence by incentive bucket

,agent,bucket_label,divergence_rate_pct,divergence_se_pct,delta_vs_baseline_pct_points,divergent_trials,total_trials,n_models,n_scenarios
0,alpha,Alignment-Inducing,39.867,1.265,37.059,598,1500,10,3
1,alpha,Baseline,2.807,0.604,0.000,21,748,10,3
2,alpha,Persona-Reinforcing,0.534,0.188,-2.274,8,1499,10,3
3,beta,Alignment-Inducing,0.600,0.200,0.333,9,1499,10,3
4,beta,Baseline,0.267,0.189,0.000,2,749,10,3
5,beta,Persona-Reinforcing,2.603,0.412,2.336,39,1498,10,3


### 1B. Agent-specific model-balanced divergence by incentive bucket

,agent,incentive_bucket,model_balanced_divergence_rate_pct,model_balanced_divergence_se_pct,model_divergence_rate_std,n_models
0,alpha,alignment_inducing,39.867,9.939,0.314,10
1,alpha,baseline,2.814,1.298,0.041,10
2,alpha,persona_reinforcing,0.535,0.260,0.008,10
3,beta,alignment_inducing,0.600,0.460,0.015,10
4,beta,baseline,0.267,0.178,0.006,10
5,beta,persona_reinforcing,2.605,1.307,0.041,10


### 1C. P Divergence by model and incentive bucket (% divergent), split by agent

incentive_bucket                            alignment_inducing  baseline  \
model                                agent                                 
anthropic/claude-opus-4.6            alpha               9.333     0.000   
                                     beta                0.000     1.333   
deepseek/deepseek-v3.2               alpha              20.667     0.000   
                                     beta                0.667     0.000   
google/gemini-3.1-flash-lite-preview alpha              15.333    12.000   
                                     beta                0.000     0.000   
google/gemini-3.1-pro-preview        alpha              91.333     0.000   
                                     beta                0.000     0.000   
minimax/minimax-m2.7                 alpha              17.333     5.333   
                                     beta                0.667     1.333   
openai/gpt-5.4                       alpha              63.333     0.000   
                                     beta                0.000     0.000   
openai/gpt-oss-120b                  alpha              12.667     5.405   
                                     beta                4.667     0.000   
qwen/qwen3.5-397b-a17b               alpha              22.667     0.000   
                                     beta                0.000     0.000   
x-ai/grok-4                          alpha              78.000     0.000   
                                     beta                0.000     0.000   
z-ai/glm-5                           alpha              68.000     5.405   
                                     beta                0.000     0.000   

incentive_bucket                            persona_reinforcing  
model                                agent                       
anthropic/claude-opus-4.6            alpha                0.000  
                                     beta                 2.000  
deepseek/deepseek-v3.2               alpha                0.667  
                                     beta                 0.000  
google/gemini-3.1-flash-lite-preview alpha                0.000  
                                     beta                 0.000  
google/gemini-3.1-pro-preview        alpha                0.000  
                                     beta                 3.333  
minimax/minimax-m2.7                 alpha                2.000  
                                     beta                12.667  
openai/gpt-5.4                       alpha                0.000  
                                     beta                 0.000  
openai/gpt-oss-120b                  alpha                0.667  
                                     beta                 6.711  
qwen/qwen3.5-397b-a17b               alpha                0.000  
                                     beta                 0.000  
x-ai/grok-4                          alpha                0.000  
                                     beta                 0.667  
z-ai/glm-5                           alpha                2.013  
                                     beta                 0.671

### 1D. Divergence by scenario and incentive bucket (% divergent), split by agent

incentive_bucket                     alignment_inducing  baseline  \
scenario_id                   agent                                 
faculty_manuscript_submission alpha                43.6     8.400   
                              beta                  0.0     0.402   
ngo_climate_endorsement       alpha                39.6     0.000   
                              beta                  1.6     0.400   
promotion_committee           alpha                36.4     0.000   
                              beta                  0.2     0.000   

incentive_bucket                     persona_reinforcing  
scenario_id                   agent                       
faculty_manuscript_submission alpha                1.603  
                              beta                 1.600  
ngo_climate_endorsement       alpha                0.000  
                              beta                 5.823  
promotion_committee           alpha                0.000  
                              beta                 0.400

Divergence trial count used: 7,493


In [3]:
display(Markdown("## 2) Cosine self-consistency summary (agent-specific)"))

COSINE_COLS = {
    "stance": "cosine-similarity-self-consistency-all-repeats",
    "no_stance": "cosine-similarity-self-consistency-all-repeats-no_stance",
}


def build_cosine_points(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        for variant, col in COSINE_COLS.items():
            payload = row.get(col)
            if not isinstance(payload, dict):
                continue

            for rep in payload.get("repeats") or []:
                if not isinstance(rep, dict):
                    continue
                rep_no = rep.get("repeat_number")
                for agent in ("alpha", "beta"):
                    block = rep.get(agent) or {}
                    turns = block.get("debate_turn") or []
                    vals = block.get("cosine_similarity") or []
                    for t, v in zip(turns, vals):
                        if v is None:
                            continue
                        rows.append(
                            {
                                "variant": variant,
                                "model": str(row.get("model")),
                                "scenario_id": str(row.get("scenario_id")),
                                "incentive_bucket": row.get("incentive_bucket"),
                                "agent": agent,
                                "repeat_number": rep_no,
                                "debate_turn": int(t),
                                "cosine": float(v),
                            }
                        )

    return pd.DataFrame(rows)


cosine_points = build_cosine_points(aggregate_df)
if cosine_points.empty:
    raise ValueError("No cosine points found in aggregate dataframe.")


def _q25(s: pd.Series) -> float:
    return float(s.quantile(0.25))


def _q75(s: pd.Series) -> float:
    return float(s.quantile(0.75))


cosine_overall = (
    cosine_points.groupby(["variant", "agent", "incentive_bucket"], as_index=False)
    .agg(
        mean_cosine=("cosine", "mean"),
        se_cosine=("cosine", "sem"),
        std_cosine=("cosine", "std"),
        median_cosine=("cosine", "median"),
        q25=("cosine", _q25),
        q75=("cosine", _q75),
        n_points=("cosine", "size"),
        n_models=("model", "nunique"),
        n_scenarios=("scenario_id", "nunique"),
    )
)

# Baseline deltas per (variant, agent).
cosine_overall = cosine_overall.sort_values(["variant", "agent", "incentive_bucket"], kind="stable").reset_index(drop=True)
base_map = (
    cosine_overall[cosine_overall["incentive_bucket"] == "baseline"]
    .set_index(["variant", "agent"])["mean_cosine"]
    .to_dict()
)
cosine_overall["delta_vs_baseline"] = [
    r.mean_cosine - base_map.get((r.variant, r.agent), np.nan)
    for r in cosine_overall.itertuples(index=False)
]

display(Markdown("### 2A. Agent-specific pooled cosine summary by incentive bucket"))
display(cosine_overall.round(4))

display(Markdown("### 2B. Agent-specific model-balanced cosine means by incentive bucket"))
cosine_model_balanced = (
    cosine_points.groupby(["variant", "agent", "model", "incentive_bucket"], as_index=False)
    .agg(model_mean_cosine=("cosine", "mean"))
    .groupby(["variant", "agent", "incentive_bucket"], as_index=False)
    .agg(
        model_balanced_mean=("model_mean_cosine", "mean"),
        model_balanced_se=("model_mean_cosine", "sem"),
        model_balanced_std=("model_mean_cosine", "std"),
        n_models=("model_mean_cosine", "size"),
    )
)
display(cosine_model_balanced.round(4))

display(Markdown("### 2C. No-stance cosine by model and incentive bucket, split by agent"))
cosine_ns = cosine_points[cosine_points["variant"] == "no_stance"].copy()
display(
    cosine_ns.groupby(["model", "agent", "incentive_bucket"], as_index=False)
    .agg(mean_cosine=("cosine", "mean"))
    .pivot(index=["model", "agent"], columns="incentive_bucket", values="mean_cosine")
    .round(4)
)

print(f"Cosine points used: {len(cosine_points):,}")

## 2) Cosine self-consistency summary (agent-specific)

### 2A. Agent-specific pooled cosine summary by incentive bucket

,variant,agent,incentive_bucket,mean_cosine,se_cosine,std_cosine,median_cosine,q25,q75,n_points,n_models,n_scenarios,delta_vs_baseline
0,no_stance,alpha,alignment_inducing,0.6600,0.0034,0.1308,0.6640,0.5696,0.7575,1500,10,3,-0.0701
1,no_stance,alpha,baseline,0.7301,0.0041,0.1119,0.7433,0.6583,0.8148,750,10,3,0.0000
2,no_stance,alpha,persona_reinforcing,0.7242,0.0028,0.1078,0.7332,0.6511,0.8056,1500,10,3,-0.0059
3,no_stance,beta,alignment_inducing,0.7253,0.0027,0.1063,0.7325,0.6627,0.8063,1500,10,3,-0.0131
4,no_stance,beta,baseline,0.7383,0.0041,0.1112,0.7459,0.6707,0.8225,750,10,3,0.0000
5,no_stance,beta,persona_reinforcing,0.7300,0.0027,0.1044,0.7399,0.6653,0.8061,1500,10,3,-0.0083
6,stance,alpha,alignment_inducing,0.7344,0.0028,0.1080,0.7393,0.6645,0.8128,1500,10,3,-0.0546
7,stance,alpha,baseline,0.7890,0.0034,0.0925,0.8008,0.7295,0.8600,750,10,3,0.0000
8,stance,alpha,persona_reinforcing,0.7861,0.0023,0.0891,0.7949,0.7296,0.8539,1500,10,3,-0.0030
9,stance,beta,alignment_inducing,0.8000,0.0020,0.0791,0.8083,0.7493,0.8575,1500,10,3,-0.0085


### 2B. Agent-specific model-balanced cosine means by incentive bucket

,variant,agent,incentive_bucket,model_balanced_mean,model_balanced_se,model_balanced_std,n_models
0,no_stance,alpha,alignment_inducing,0.6600,0.0214,0.0675,10
1,no_stance,alpha,baseline,0.7301,0.0180,0.0568,10
2,no_stance,alpha,persona_reinforcing,0.7242,0.0175,0.0555,10
3,no_stance,beta,alignment_inducing,0.7253,0.0148,0.0467,10
4,no_stance,beta,baseline,0.7383,0.0197,0.0624,10
5,no_stance,beta,persona_reinforcing,0.7300,0.0157,0.0496,10
6,stance,alpha,alignment_inducing,0.7344,0.0170,0.0537,10
7,stance,alpha,baseline,0.7890,0.0124,0.0391,10
8,stance,alpha,persona_reinforcing,0.7861,0.0116,0.0368,10
9,stance,beta,alignment_inducing,0.8000,0.0099,0.0312,10


### 2C. No-stance cosine by model and incentive bucket, split by agent

incentive_bucket                            alignment_inducing  baseline  \
model                                agent                                 
anthropic/claude-opus-4.6            alpha              0.6325    0.6811   
                                     beta               0.6639    0.6849   
deepseek/deepseek-v3.2               alpha              0.5986    0.6470   
                                     beta               0.6847    0.6755   
google/gemini-3.1-flash-lite-preview alpha              0.6324    0.6608   
                                     beta               0.6935    0.6849   
google/gemini-3.1-pro-preview        alpha              0.6155    0.7751   
                                     beta               0.7415    0.7785   
minimax/minimax-m2.7                 alpha              0.7204    0.7418   
                                     beta               0.7319    0.7336   
openai/gpt-5.4                       alpha              0.7380    0.7830   
                                     beta               0.7845    0.8021   
openai/gpt-oss-120b                  alpha              0.7789    0.8006   
                                     beta               0.7782    0.8061   
qwen/qwen3.5-397b-a17b               alpha              0.6509    0.7509   
                                     beta               0.7183    0.7659   
x-ai/grok-4                          alpha              0.6708    0.7783   
                                     beta               0.7852    0.8097   
z-ai/glm-5                           alpha              0.5615    0.6825   
                                     beta               0.6711    0.6423   

incentive_bucket                            persona_reinforcing  
model                                agent                       
anthropic/claude-opus-4.6            alpha               0.6528  
                                     beta                0.6814  
deepseek/deepseek-v3.2               alpha               0.6497  
                                     beta                0.6789  
google/gemini-3.1-flash-lite-preview alpha               0.6825  
                                     beta                0.7138  
google/gemini-3.1-pro-preview        alpha               0.7589  
                                     beta                0.7478  
minimax/minimax-m2.7                 alpha               0.7363  
                                     beta                0.6994  
openai/gpt-5.4                       alpha               0.7953  
                                     beta                0.7987  
openai/gpt-oss-120b                  alpha               0.7952  
                                     beta                0.7806  
qwen/qwen3.5-397b-a17b               alpha               0.7289  
                                     beta                0.7278  
x-ai/grok-4                          alpha               0.7647  
                                     beta                0.8001  
z-ai/glm-5                           alpha               0.6772  
                                     beta                0.6716

Cosine points used: 15,000


In [4]:
display(Markdown("## 3) NLI summary (entailment / neutral / contradiction, agent-specific where available)"))

NLI_SOURCES = {
    "self_no_stance": "nli-self-consistency-all-repeats-no_stance",
    "self_stance": "nli-self-consistency-all-repeats",
    "cross_alignment": "nli-cross-agent-alignment-all-repeats",
}


def build_nli_points(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        for source, col in NLI_SOURCES.items():
            payload = row.get(col)
            if not isinstance(payload, dict):
                continue

            for rep in payload.get("repeats") or []:
                if not isinstance(rep, dict):
                    continue
                rep_no = rep.get("repeat_number")

                if source.startswith("self"):
                    channel_keys = ("alpha", "beta")
                else:
                    channel_keys = ("public utterances", "private reflections")

                for channel in channel_keys:
                    block = rep.get(channel) or {}
                    turns = block.get("debate_turns") or []
                    probs = block.get("nli_probabilities") or []
                    ordering = block.get("nli_tuple_ordering") or ()

                    # Agent-specific where possible (self sources).
                    agent = channel if source.startswith("self") else pd.NA

                    for t, p in zip(turns, probs):
                        if not isinstance(p, (tuple, list)):
                            continue
                        mapped = {lbl: float(val) for lbl, val in zip(ordering, p)}
                        ent = mapped.get("entailment", np.nan)
                        neu = mapped.get("neutral", np.nan)
                        con = mapped.get("contradiction", np.nan)
                        if np.isnan(ent) and np.isnan(neu) and np.isnan(con):
                            continue

                        dominant_label = max(
                            ("entailment", "neutral", "contradiction"),
                            key=lambda k: -1.0 if np.isnan(mapped.get(k, np.nan)) else mapped.get(k, np.nan),
                        )

                        rows.append(
                            {
                                "source": source,
                                "model": str(row.get("model")),
                                "scenario_id": str(row.get("scenario_id")),
                                "incentive_bucket": row.get("incentive_bucket"),
                                "agent": agent,
                                "channel": channel,
                                "repeat_number": rep_no,
                                "debate_turn": int(t),
                                "entailment": ent,
                                "neutral": neu,
                                "contradiction": con,
                                "dominant_label": dominant_label,
                            }
                        )

    return pd.DataFrame(rows)


nli_points = build_nli_points(aggregate_df)
if nli_points.empty:
    raise ValueError("No NLI points found in aggregate dataframe.")

# Self-consistency NLI is alpha/beta specific.
nli_self = nli_points[nli_points["source"].isin(["self_no_stance", "self_stance"])].copy()

display(Markdown("### 3A. Mean NLI probabilities by source, agent, and incentive bucket"))
nli_mean = (
    nli_self.groupby(["source", "agent", "incentive_bucket"], as_index=False)
    .agg(
        entailment=("entailment", "mean"),
        entailment_se=("entailment", "sem"),
        neutral=("neutral", "mean"),
        neutral_se=("neutral", "sem"),
        contradiction=("contradiction", "mean"),
        contradiction_se=("contradiction", "sem"),
        n_points=("entailment", "size"),
        n_models=("model", "nunique"),
        n_scenarios=("scenario_id", "nunique"),
    )
)
for c in ("entailment", "entailment_se", "neutral", "neutral_se", "contradiction", "contradiction_se"):
    nli_mean[c] = 100.0 * nli_mean[c]
display(nli_mean.round(4))

display(Markdown("### 3B. Dominant-label share by source, agent, and incentive bucket"))
dominant = (
    nli_self.groupby(["source", "agent", "incentive_bucket", "dominant_label"], as_index=False)
    .size()
    .rename(columns={"size": "n"})
)
dominant["share"] = dominant["n"] / dominant.groupby(["source", "agent", "incentive_bucket"])["n"].transform("sum")

dominant_wide = dominant.pivot(
    index=["source", "agent", "incentive_bucket"],
    columns="dominant_label",
    values="share",
).fillna(0.0)

display((100.0 * dominant_wide).round(3))

display(Markdown("### 3C. Baseline deltas for contradiction/entailment by source and agent (mean prob points)"))
base_nli = (
    nli_mean[nli_mean["incentive_bucket"] == "baseline"]
    .set_index(["source", "agent"])[["entailment", "contradiction"]]
    .to_dict(orient="index")
)
rows = []
for r in nli_mean.itertuples(index=False):
    b = base_nli.get((r.source, r.agent), {})
    rows.append(
        {
            "source": r.source,
            "agent": r.agent,
            "incentive_bucket": r.incentive_bucket,
            "delta_entailment_vs_baseline": r.entailment - b.get("entailment", np.nan),
            "delta_contradiction_vs_baseline": r.contradiction - b.get("contradiction", np.nan),
        }
    )
display(pd.DataFrame(rows).round(4))

display(Markdown("### 3D. Cross-agent alignment NLI (channel-level, not alpha/beta specific)"))
nli_cross = nli_points[nli_points["source"] == "cross_alignment"].copy()
if not nli_cross.empty:
    cross_mean = (
        nli_cross.groupby(["channel", "incentive_bucket"], as_index=False)
        .agg(
            entailment=("entailment", "mean"),
            entailment_se=("entailment", "sem"),
            neutral=("neutral", "mean"),
            neutral_se=("neutral", "sem"),
            contradiction=("contradiction", "mean"),
            contradiction_se=("contradiction", "sem"),
            n_points=("entailment", "size"),
        )
    )
    for c in ("entailment", "entailment_se", "neutral", "neutral_se", "contradiction", "contradiction_se"):
        cross_mean[c] = 100.0 * cross_mean[c]
    display(cross_mean.round(4))
else:
    print("No cross_alignment NLI rows found.")

print(f"NLI points used: {len(nli_points):,}")

## 3) NLI summary (entailment / neutral / contradiction, agent-specific where available)

### 3A. Mean NLI probabilities by source, agent, and incentive bucket

,source,agent,incentive_bucket,entailment,entailment_se,neutral,neutral_se,contradiction,contradiction_se,n_points,n_models,n_scenarios
0,self_no_stance,alpha,alignment_inducing,15.2951,0.6639,65.2584,0.8980,19.4465,0.8571,1500,10,3
1,self_no_stance,alpha,baseline,32.7128,1.1321,65.2024,1.1222,2.0848,0.3780,750,10,3
2,self_no_stance,alpha,persona_reinforcing,29.5197,0.7670,69.1592,0.7653,1.3212,0.1736,1500,10,3
3,self_no_stance,beta,alignment_inducing,24.6517,0.7662,73.6487,0.7771,1.6996,0.2464,1500,10,3
4,self_no_stance,beta,baseline,32.3441,1.2120,66.4629,1.2067,1.1930,0.2275,750,10,3
5,self_no_stance,beta,persona_reinforcing,27.6168,0.7980,69.7235,0.8157,2.6598,0.3390,1500,10,3
6,self_stance,alpha,alignment_inducing,14.8632,0.6685,45.6062,1.0990,39.5305,1.2292,1500,10,3
7,self_stance,alpha,baseline,32.8054,1.1426,63.3804,1.1814,3.8142,0.6421,750,10,3
8,self_stance,alpha,persona_reinforcing,29.3570,0.7648,69.2426,0.7718,1.4003,0.2254,1500,10,3
9,self_stance,beta,alignment_inducing,24.0557,0.7505,73.9091,0.7685,2.0352,0.2774,1500,10,3


### 3B. Dominant-label share by source, agent, and incentive bucket

dominant_label                            contradiction  entailment  neutral
source         agent incentive_bucket                                       
self_no_stance alpha alignment_inducing          18.667       9.867   71.467
                     baseline                     0.933      24.267   74.800
                     persona_reinforcing          0.733      21.400   77.867
               beta  alignment_inducing           1.267      16.333   82.400
                     baseline                     0.533      26.267   73.200
                     persona_reinforcing          2.067      19.800   78.133
self_stance    alpha alignment_inducing          39.933       9.733   50.333
                     baseline                     3.333      25.733   70.933
                     persona_reinforcing          0.867      20.800   78.333
               beta  alignment_inducing           1.533      16.133   82.333
                     baseline                     1.067      26.000   72.933
                     persona_reinforcing          3.200      19.200   77.600

### 3C. Baseline deltas for contradiction/entailment by source and agent (mean prob points)

,source,agent,incentive_bucket,delta_entailment_vs_baseline,delta_contradiction_vs_baseline
0,self_no_stance,alpha,alignment_inducing,-17.4177,17.3617
1,self_no_stance,alpha,baseline,0.0000,0.0000
2,self_no_stance,alpha,persona_reinforcing,-3.1931,-0.7636
3,self_no_stance,beta,alignment_inducing,-7.6924,0.5066
4,self_no_stance,beta,baseline,0.0000,0.0000
5,self_no_stance,beta,persona_reinforcing,-4.7273,1.4667
6,self_stance,alpha,alignment_inducing,-17.9422,35.7163
7,self_stance,alpha,baseline,0.0000,0.0000
8,self_stance,alpha,persona_reinforcing,-3.4484,-2.4138
9,self_stance,beta,alignment_inducing,-7.3271,0.2628


### 3D. Cross-agent alignment NLI (channel-level, not alpha/beta specific)

,channel,incentive_bucket,entailment,entailment_se,neutral,neutral_se,contradiction,contradiction_se,n_points
0,private reflections,alignment_inducing,2.6914,0.3184,14.9621,0.8389,82.3465,0.9413,1500
1,private reflections,baseline,0.6993,0.2494,2.5736,0.4739,96.7270,0.5787,750
2,private reflections,persona_reinforcing,0.4144,0.0939,3.4677,0.3953,96.1179,0.4403,1500
3,public utterances,alignment_inducing,12.0028,0.5842,36.8585,1.0698,51.1387,1.2676,1500
4,public utterances,baseline,0.5997,0.1611,2.4157,0.3935,96.9847,0.4831,750
5,public utterances,persona_reinforcing,0.5291,0.1045,2.3464,0.2697,97.1245,0.3288,1500


NLI points used: 22,500


In [5]:
display(Markdown("## 4) Paper-ready textual snippets (agent-specific)"))

for agent in ("alpha", "beta"):
    print("=" * 72)
    print(f"Agent: {agent}")
    print("=" * 72)

    # Divergence snippet
    _div = div_overall[div_overall["agent"] == agent].set_index("incentive_bucket")
    try:
        pr = float(_div.loc["persona_reinforcing", "divergence_rate_pct"])
        bl = float(_div.loc["baseline", "divergence_rate_pct"])
        ai = float(_div.loc["alignment_inducing", "divergence_rate_pct"])
        pr_se = float(_div.loc["persona_reinforcing", "divergence_se_pct"])
        bl_se = float(_div.loc["baseline", "divergence_se_pct"])
        ai_se = float(_div.loc["alignment_inducing", "divergence_se_pct"])

        print("Divergence (pooled over models/scenarios/repeats/turns):")
        print(f"- Persona-Reinforcing: {pr:.3f}% ± {pr_se:.3f}")
        print(f"- Baseline:            {bl:.3f}% ± {bl_se:.3f}")
        print(f"- Alignment-Inducing:  {ai:.3f}% ± {ai_se:.3f}")
        print(f"- Delta (Persona-Reinforcing - Baseline): {pr - bl:+.3f} percentage points")
        print(f"- Delta (Alignment-Inducing - Baseline):  {ai - bl:+.3f} percentage points")
    except Exception:
        print("Could not produce divergence snippet for all three canonical buckets.")

    print()

    # Cosine snippet (no_stance)
    _cos = cosine_overall[
        (cosine_overall["variant"] == "no_stance") & (cosine_overall["agent"] == agent)
    ].set_index("incentive_bucket")
    try:
        pr = float(_cos.loc["persona_reinforcing", "mean_cosine"])
        bl = float(_cos.loc["baseline", "mean_cosine"])
        ai = float(_cos.loc["alignment_inducing", "mean_cosine"])
        pr_se = float(_cos.loc["persona_reinforcing", "se_cosine"])
        bl_se = float(_cos.loc["baseline", "se_cosine"])
        ai_se = float(_cos.loc["alignment_inducing", "se_cosine"])
        print("Cosine self-consistency (no_stance):")
        print(f"- Persona-Reinforcing: {pr:.4f} ± {pr_se:.4f}")
        print(f"- Baseline:            {bl:.4f} ± {bl_se:.4f}")
        print(f"- Alignment-Inducing:  {ai:.4f} ± {ai_se:.4f}")
        print(f"- Delta (Persona-Reinforcing - Baseline): {pr - bl:+.4f}")
        print(f"- Delta (Alignment-Inducing - Baseline):  {ai - bl:+.4f}")
    except Exception:
        print("Could not produce cosine snippet for all three canonical buckets.")

    print()

    # NLI snippet (self_no_stance)
    _nli = nli_mean[
        (nli_mean["source"] == "self_no_stance") & (nli_mean["agent"] == agent)
    ].set_index("incentive_bucket")
    try:
        for b in BUCKET_ORDER:
            lbl = BUCKET_LABEL.get(b, b)
            ent = float(_nli.loc[b, "entailment"])
            neu = float(_nli.loc[b, "neutral"])
            con = float(_nli.loc[b, "contradiction"])
            ent_se = float(_nli.loc[b, "entailment_se"])
            neu_se = float(_nli.loc[b, "neutral_se"])
            con_se = float(_nli.loc[b, "contradiction_se"])
            print(
                f"NLI self_no_stance ({lbl}): "
                f"entail={ent:.3f}% ± {ent_se:.3f}, "
                f"neutral={neu:.3f}% ± {neu_se:.3f}, "
                f"contradiction={con:.3f}% ± {con_se:.3f}"
            )
    except Exception:
        print("Could not produce NLI snippet for all three canonical buckets.")

    print()

## 4) Paper-ready textual snippets (agent-specific)

Agent: alpha
Divergence (pooled over models/scenarios/repeats/turns):
- Persona-Reinforcing: 0.534% ± 0.188
- Baseline:            2.807% ± 0.604
- Alignment-Inducing:  39.867% ± 1.265
- Delta (Persona-Reinforcing - Baseline): -2.274 percentage points
- Delta (Alignment-Inducing - Baseline):  +37.059 percentage points

Cosine self-consistency (no_stance):
- Persona-Reinforcing: 0.7242 ± 0.0028
- Baseline:            0.7301 ± 0.0041
- Alignment-Inducing:  0.6600 ± 0.0034
- Delta (Persona-Reinforcing - Baseline): -0.0059
- Delta (Alignment-Inducing - Baseline):  -0.0701

NLI self_no_stance (Persona-Reinforcing): entail=29.520% ± 0.767, neutral=69.159% ± 0.765, contradiction=1.321% ± 0.174
NLI self_no_stance (Baseline): entail=32.713% ± 1.132, neutral=65.202% ± 1.122, contradiction=2.085% ± 0.378
NLI self_no_stance (Alignment-Inducing): entail=15.295% ± 0.664, neutral=65.258% ± 0.898, contradiction=19.447% ± 0.857

Agent: beta
Divergence (pooled over models/scenarios/repeats/turns):
- Per

## 5) Data integrity and coverage audit

This section is a non-plot QC report to verify denominator consistency and identify missing/unpaired turns.

It includes:

- experiment inventory (models, scenarios, incentive buckets),
- decision coverage audit (paired turns vs dropped turns),
- exact rows with unpaired public/private turn lengths,
- model/scenario-level coverage checks,
- quick point-count checks for cosine and NLI sources.

In [6]:
display(Markdown("### 5A. Experiment inventory"))

inv = pd.DataFrame(
    [
        {
            "n_experiment_rows": len(aggregate_df),
            "n_models": aggregate_df["model"].nunique(),
            "n_scenarios": aggregate_df["scenario_id"].nunique(),
            "n_incentive_pairs": aggregate_df[["incentive_direction", "incentive_type"]].drop_duplicates().shape[0],
            "repeat_count_values": sorted(pd.Series(aggregate_df["repeat_count"]).dropna().unique().tolist()),
        }
    ]
)
display(inv)


display(Markdown("### 5B. Decision pairing audit (public vs private turns)"))

DECISION_COL = "decision-self-consistency-all-repeats"

audit_rows = []
for _, row in aggregate_df.iterrows():
    payload = row.get(DECISION_COL)
    if not isinstance(payload, dict):
        continue

    for rep in payload.get("repeats") or []:
        if not isinstance(rep, dict):
            continue
        rep_no = rep.get("repeat_number")

        for agent in ("alpha", "beta"):
            block = rep.get(agent) or {}
            pub_block = block.get("public") or {}
            prv_block = block.get("private") or {}

            pub = pub_block.get("decisions") or []
            prv = prv_block.get("decisions") or []
            pub_turns = pub_block.get("turns") or []
            prv_turns = prv_block.get("turns") or []

            paired = min(len(pub), len(prv))
            unpaired = abs(len(pub) - len(prv))

            n_none = 0
            n_valid = 0
            for a, b in zip(pub, prv):
                if a is None or b is None:
                    n_none += 1
                else:
                    n_valid += 1

            audit_rows.append(
                {
                    "incentive_bucket": row.get("incentive_bucket"),
                    "incentive_direction": row.get("incentive_direction"),
                    "incentive_type": row.get("incentive_type"),
                    "model": str(row.get("model")),
                    "scenario_id": str(row.get("scenario_id")),
                    "repeat_number": rep_no,
                    "agent": agent,
                    "len_pub": len(pub),
                    "len_prv": len(prv),
                    "len_pub_turns": len(pub_turns),
                    "len_prv_turns": len(prv_turns),
                    "paired_turns": paired,
                    "unpaired_turns": unpaired,
                    "none_in_paired_turns": n_none,
                    "valid_paired_turns": n_valid,
                }
            )


decision_audit = pd.DataFrame(audit_rows)
if decision_audit.empty:
    raise ValueError("Decision audit table is empty.")

bucket_cov = (
    decision_audit.groupby("incentive_bucket", as_index=False)
    .agg(
        agent_repeat_blocks=("agent", "size"),
        paired_turns_total=("paired_turns", "sum"),
        valid_paired_turns_total=("valid_paired_turns", "sum"),
        dropped_unpaired_turns_total=("unpaired_turns", "sum"),
        dropped_none_turns_total=("none_in_paired_turns", "sum"),
        mean_len_pub=("len_pub", "mean"),
        mean_len_prv=("len_prv", "mean"),
    )
)
bucket_cov["bucket_label"] = bucket_cov["incentive_bucket"].map(BUCKET_LABEL).fillna(bucket_cov["incentive_bucket"])

display(bucket_cov.round(4))


display(Markdown("### 5C. Exact rows with unpaired turns (where public/private lengths differ)"))

unpaired = decision_audit[decision_audit["unpaired_turns"] > 0].copy()
unpaired = unpaired.sort_values(
    ["incentive_bucket", "model", "scenario_id", "repeat_number", "agent"],
    kind="stable",
)

if unpaired.empty:
    print("No unpaired turns found.")
else:
    display(
        unpaired[
            [
                "incentive_bucket",
                "incentive_direction",
                "incentive_type",
                "model",
                "scenario_id",
                "repeat_number",
                "agent",
                "len_pub",
                "len_prv",
                "unpaired_turns",
            ]
        ]
    )


display(Markdown("### 5D. Coverage by model x scenario"))

model_scen_cov = (
    decision_audit.groupby(["model", "scenario_id"], as_index=False)
    .agg(
        blocks=("agent", "size"),
        paired_turns_total=("paired_turns", "sum"),
        valid_paired_turns_total=("valid_paired_turns", "sum"),
        dropped_unpaired_turns_total=("unpaired_turns", "sum"),
        dropped_none_turns_total=("none_in_paired_turns", "sum"),
    )
)

display(model_scen_cov.sort_values(["model", "scenario_id"], kind="stable"))


def _count_cosine_points(col: str) -> dict:
    n_payload_rows = 0
    n_points = 0
    n_missing = 0
    for _, row in aggregate_df.iterrows():
        payload = row.get(col)
        if not isinstance(payload, dict):
            continue
        n_payload_rows += 1
        for rep in payload.get("repeats") or []:
            if not isinstance(rep, dict):
                continue
            for agent in ("alpha", "beta"):
                vals = (rep.get(agent) or {}).get("cosine_similarity") or []
                for v in vals:
                    if v is None:
                        n_missing += 1
                    else:
                        n_points += 1
    return {
        "metric": col,
        "payload_rows": n_payload_rows,
        "valid_points": n_points,
        "none_points": n_missing,
    }


def _count_nli_points(col: str, mode: str) -> dict:
    n_payload_rows = 0
    n_points = 0
    n_missing = 0

    if mode == "self":
        keys = ("alpha", "beta")
    else:
        keys = ("public utterances", "private reflections")

    for _, row in aggregate_df.iterrows():
        payload = row.get(col)
        if not isinstance(payload, dict):
            continue
        n_payload_rows += 1
        for rep in payload.get("repeats") or []:
            if not isinstance(rep, dict):
                continue
            for k in keys:
                probs = (rep.get(k) or {}).get("nli_probabilities") or []
                for triplet in probs:
                    if triplet is None:
                        n_missing += 1
                    else:
                        n_points += 1

    return {
        "metric": col,
        "payload_rows": n_payload_rows,
        "valid_points": n_points,
        "none_points": n_missing,
    }


display(Markdown("### 5E. Quick point-count completeness checks (cosine + NLI)"))

qc_rows = []
qc_rows.append(_count_cosine_points("cosine-similarity-self-consistency-all-repeats"))
qc_rows.append(_count_cosine_points("cosine-similarity-self-consistency-all-repeats-no_stance"))
qc_rows.append(_count_nli_points("nli-self-consistency-all-repeats", mode="self"))
qc_rows.append(_count_nli_points("nli-self-consistency-all-repeats-no_stance", mode="self"))
qc_rows.append(_count_nli_points("nli-cross-agent-alignment-all-repeats", mode="cross"))

display(pd.DataFrame(qc_rows))

print("Done: integrity/coverage audit generated.")

### 5A. Experiment inventory

,n_experiment_rows,n_models,n_scenarios,n_incentive_pairs,repeat_count_values
0,150,10,3,5,[5]


### 5B. Decision pairing audit (public vs private turns)

,incentive_bucket,agent_repeat_blocks,paired_turns_total,valid_paired_turns_total,dropped_unpaired_turns_total,dropped_none_turns_total,mean_len_pub,mean_len_prv,bucket_label
0,alignment_inducing,600,2999,2999,1,0,4.9983,5.0000,Alignment-Inducing
1,baseline,300,1497,1497,3,0,5.0000,4.9900,Baseline
2,persona_reinforcing,600,2997,2997,3,0,4.9983,4.9967,Persona-Reinforcing


### 5C. Exact rows with unpaired turns (where public/private lengths differ)

...

### 5D. Coverage by model x scenario

,model,scenario_id,blocks,paired_turns_total,valid_paired_turns_total,dropped_unpaired_turns_total,dropped_none_turns_total
0,anthropic/claude-opus-4.6,faculty_manuscript_submission,50,250,250,0,0
1,anthropic/claude-opus-4.6,ngo_climate_endorsement,50,250,250,0,0
2,anthropic/claude-opus-4.6,promotion_committee,50,250,250,0,0
3,deepseek/deepseek-v3.2,faculty_manuscript_submission,50,250,250,0,0
4,deepseek/deepseek-v3.2,ngo_climate_endorsement,50,250,250,0,0
5,deepseek/deepseek-v3.2,promotion_committee,50,250,250,0,0
6,google/gemini-3.1-flash-lite-preview,faculty_manuscript_submission,50,250,250,0,0
7,google/gemini-3.1-flash-lite-preview,ngo_climate_endorsement,50,250,250,0,0
8,google/gemini-3.1-flash-lite-preview,promotion_committee,50,249,249,1,0
9,google/gemini-3.1-pro-preview,faculty_manuscript_submission,50,250,250,0,0


### 5E. Quick point-count completeness checks (cosine + NLI)

,metric,payload_rows,valid_points,none_points
0,cosine-similarity-self-consistency-all-repeats,150,7500,0
1,cosine-similarity-self-consistency-all-repeats...,150,7500,0
2,nli-self-consistency-all-repeats,150,7500,0
3,nli-self-consistency-all-repeats-no_stance,150,7500,0
4,nli-cross-agent-alignment-all-repeats,150,7500,0


Done: integrity/coverage audit generated.


In [ ]:
display(Markdown("## 1E) Turn-aligned divergence trial audit (fixes positional zip pairing)"))

DECISION_COL = "decision-self-consistency-all-repeats"


def _turn_to_decision_map(channel_payload: dict) -> dict[int, int]:
    decisions = list((channel_payload or {}).get("decisions") or [])
    turns = list((channel_payload or {}).get("turns") or [])
    if not turns:
        turns = list(range(1, len(decisions) + 1))

    out: dict[int, int] = {}
    for i, decision in enumerate(decisions):
        if decision is None:
            continue
        turn = int(turns[i]) if i < len(turns) else (i + 1)
        out[turn] = int(decision)
    return out


def build_turn_aligned_divergence_trials(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    trial_rows = []
    coverage_rows = []

    for _, row in df.iterrows():
        payload = row.get(DECISION_COL)
        if not isinstance(payload, dict):
            continue

        for rep in payload.get("repeats") or []:
            if not isinstance(rep, dict):
                continue
            rep_no = rep.get("repeat_number")
            for agent in ("alpha", "beta"):
                block = rep.get(agent) or rep.get(agent.capitalize()) or {}
                pub_map = _turn_to_decision_map((block.get("public") or {}))
                prv_map = _turn_to_decision_map((block.get("private") or {}))

                pub_turns = set(pub_map)
                prv_turns = set(prv_map)
                paired_turns = sorted(pub_turns & prv_turns)
                all_turns = sorted(pub_turns | prv_turns)

                coverage_rows.append(
                    {
                        "model": str(row.get("model")),
                        "scenario_id": str(row.get("scenario_id")),
                        "incentive_bucket": row.get("incentive_bucket"),
                        "incentive_direction": row.get("incentive_direction"),
                        "incentive_type": row.get("incentive_type"),
                        "agent": agent,
                        "repeat_number": rep_no,
                        "expected_turns": len(all_turns),
                        "paired_turns": len(paired_turns),
                        "missing_pub_turns": len(prv_turns - pub_turns),
                        "missing_prv_turns": len(pub_turns - prv_turns),
                        "missing_either_turns": len(all_turns) - len(paired_turns),
                    }
                )

                for turn in paired_turns:
                    trial_rows.append(
                        {
                            "model": str(row.get("model")),
                            "scenario_id": str(row.get("scenario_id")),
                            "incentive_bucket": row.get("incentive_bucket"),
                            "incentive_direction": row.get("incentive_direction"),
                            "incentive_type": row.get("incentive_type"),
                            "agent": agent,
                            "repeat_number": rep_no,
                            "debate_turn": int(turn),
                            "divergent": int(pub_map[turn] != prv_map[turn]),
                        }
                    )

    return pd.DataFrame(trial_rows), pd.DataFrame(coverage_rows)


div_trials_aligned, div_cov_aligned = build_turn_aligned_divergence_trials(aggregate_df)

coverage_overall_aligned = (
    div_cov_aligned.groupby(["agent", "incentive_bucket"], as_index=False)
    .agg(
        total_trials_expected=("expected_turns", "sum"),
        total_trials_paired=("paired_turns", "sum"),
        missing_trials=("missing_either_turns", "sum"),
        missing_pub_trials=("missing_pub_turns", "sum"),
        missing_prv_trials=("missing_prv_turns", "sum"),
    )
)

div_overall_aligned = (
    div_trials_aligned.groupby(["agent", "incentive_bucket"], as_index=False)
    .agg(
        divergence_rate=("divergent", "mean"),
        divergence_se=("divergent", "sem"),
        divergent_trials=("divergent", "sum"),
        n_models=("model", "nunique"),
        n_scenarios=("scenario_id", "nunique"),
    )
    .merge(coverage_overall_aligned, on=["agent", "incentive_bucket"], how="left")
)

div_overall_aligned["bucket_label"] = (
    div_overall_aligned["incentive_bucket"].map(BUCKET_LABEL).fillna(div_overall_aligned["incentive_bucket"])
)
div_overall_aligned["divergence_rate_pct"] = 100.0 * div_overall_aligned["divergence_rate"]
div_overall_aligned["divergence_se_pct"] = 100.0 * div_overall_aligned["divergence_se"]

base_map_aligned = (
    div_overall_aligned[div_overall_aligned["incentive_bucket"] == "baseline"]
    .set_index("agent")["divergence_rate_pct"]
    .to_dict()
)
div_overall_aligned["delta_vs_baseline_pct_points"] = [
    r.divergence_rate_pct - base_map_aligned.get(r.agent, np.nan)
    for r in div_overall_aligned.itertuples(index=False)
]

div_overall_aligned = div_overall_aligned.sort_values(["agent", "incentive_bucket"], kind="stable").reset_index(drop=True)

display(
    div_overall_aligned[
        [
            "agent",
            "bucket_label",
            "divergence_rate_pct",
            "divergence_se_pct",
            "delta_vs_baseline_pct_points",
            "divergent_trials",
            "total_trials_expected",
            "total_trials_paired",
            "missing_trials",
            "missing_pub_trials",
            "missing_prv_trials",
            "n_models",
            "n_scenarios",
        ]
    ].round(3)
)

print(f"Turn-aligned paired trials used: {len(div_trials_aligned):,}")
print(f"Expected trial slots (public/private union): {int(div_cov_aligned['expected_turns'].sum()):,}")
